# GEE Monthly Mosaic — Quickstart

Generate cloud-free Sentinel-2 monthly thumbnail URLs for the last 12 months.

## Setup

In [ ]:
import ee
from gee_monthly_mosaic import MonthlyMosaicBuilder

ee.Initialize(project='mapbiomas')
print('GEE initialized ✓')

: 

## 1. Define area of interest

In [ ]:
# Point with 1km buffer
geometry = ee.Geometry.Point(-57.87255, -10.03289).buffer(1000)

# Alternative: polygon from WKT
# from shapely import wkt
# shape = wkt.loads("POLYGON (...)")
# geometry = ee.Geometry.Polygon(shape.exterior.coords[:])

print(f'AOI bounds: {geometry.bounds().getInfo()}')

## 2. Build and get URLs

In [ ]:
builder = MonthlyMosaicBuilder(
    geometry=geometry,
    end_date="2024-12-31",
    n_months=12,
)

print("Generating URLs (parallel, 6 workers)...")
results = builder.get_urls()

print(f"\nGenerated {len(results)} monthly thumbnails:\n")
for r in results:
    status = '✓' if r['url'] else '✗ (no data)'
    print(f"  {r['month']}  n_images={r['n_images']:2d}  {status}")

## 3. Visualize (optional, requires geemap)

In [ ]:
import requests
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image

# Display all 12 months (3 rows × 4 cols)
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.patch.set_facecolor('#1a1a2e')

for idx, (ax, r) in enumerate(zip(axes.flat, results)):
    ax.set_facecolor('#0d0d0d')
    
    if r['url']:
        resp = requests.get(r['url'], timeout=30)
        img = Image.open(BytesIO(resp.content)).convert('RGB')
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, color='#ff6b6b')
    
    ax.set_title(f"{r['month']} (n={r['n_images']})", color='white', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Try different composites

In [ ]:
# Compare all 4 composites for a single month
import requests
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
from gee_monthly_mosaic import COMPOSITES

# Pick a month with good data (August typically has less cloud cover)
target_month_date = '2024-08-31'
target_month_label = '2024-08'

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle(f'All Composites — {target_month_label}', color='white', fontsize=14, y=1.02)

for idx, (ax, composite_name) in enumerate(zip(axes, COMPOSITES.keys())):
    # Build URL for this composite
    builder = MonthlyMosaicBuilder(
        geometry=geometry,
        end_date=target_month_date,
        n_months=1,
        composite=composite_name,
        paint_geometry=False,  # no overlay for clean comparison
    )
    urls = builder.get_urls(workers=1)
    r = urls[0]

    ax.set_facecolor('#0d0d0d')

    if r['url']:
        resp = requests.get(r['url'], timeout=30)
        img = Image.open(BytesIO(resp.content)).convert('RGB')
        ax.imshow(img)
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                transform=ax.transAxes, color='#ff6b6b', fontsize=12)

    ax.set_title(composite_name, color='white', fontsize=11, pad=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

print(f'Composites for {target_month_label}: {sum(1 for u in urls if u["url"])} / 4 available')

## 5. Export full-resolution mosaics (optional)

In [ ]:
# Export to Google Drive (all bands, 30m resolution)
# task_ids = builder.export_to_drive(folder="mapbiomas-mosaics", scale=30)
# print(f"Submitted {len(task_ids)} export tasks:")
# for tid in task_ids:
#     print(f"  {tid}")

# Monitor in the GEE Tasks panel (https://code.earthengine.google.com/tasks)

print("Export feature (commented out by default)")